#### First some imports:

In [ ]:
from agent import DatabaseAgent
import duckdb
import argparse
import logging
from datetime import datetime
import matplotlib.pyplot as plt
import os
from pprint import pprint
from prompts import build_sql_prompt, build_presentation_prompt_df, build_presentation_prompt_short
from prompts import build_presentation_type_prompt, build_chart_prompt, CHART_SCHEMA
from plots import build_plot
import numpy as np
from eval import PLOT_TEST_CASES, PLOT_TEST_CASES2, TEST_CASES3
import pandas as pd

#### Open database and instantiate agent

In [2]:
def log(message):
    pass

db_path = '../data/jaffle_shop.duckdb'
conn = duckdb.connect(db_path)
agent = DatabaseAgent(conn, log=log, temperature=0)
special_columns = [{"table":"customers", "col": "loyalty_tier"},
                   {"table":"orders", "col": "status"}]
pprint(agent.get_special_columns_content(special_columns))

[{'column': 'customers.loyalty_tier',
  'values': ['bronze', 'silver', 'platinum', 'gold']},
 {'column': 'orders.status', 'values': ['returned', 'completed', 'cancelled']}]


#### Loop through all test cases:

In [3]:
test_case_results = []
for test_case_result in TEST_CASES3:
    test_dict = test_case_result.copy()
    print(f"Running test case: {test_case_result['id']}")
    question = test_case_result["question"]
    response = agent(question)
    #print("Response:")
    #pprint(response)
    #print("\n\n")
    test_dict["response"] = response
    test_case_results.append(test_dict)


Running test case: tc_01
Running test case: tc_02
Running test case: tc_03
Running test case: tc_04
Running test case: tc_05
Running test case: tc_06
Running test case: tc_07
Running test case: tc_08
Running test case: tc_09
Running test case: tc_10


In [ ]:
# Optional: print the results in a more readable format
# print(test_case_results)

We build a small evaluator, which takes the sum over all numeric columns in the data set, to compare the results from our LLM, with some golden standard result. 

In [16]:
SKIP_COLUMNS = {
    "customer_id",
    "created_at",
    "order_id",
    "product_id",
    "order_date",
    "order_item_id",
    
}

def compare_by_numeric_sums_ignore_names(
    actual: pd.DataFrame,
    expected: pd.DataFrame,
    atol: float = 1e-6,
    rtol: float = 1e-6,
) -> tuple[bool, dict]:
    def get_sorted_numeric_sums(df: pd.DataFrame) -> np.ndarray:
        numeric_cols = [
            c for c in df.columns
            if c not in SKIP_COLUMNS and pd.api.types.is_numeric_dtype(df[c])
        ]
        sums = df[numeric_cols].sum(axis=0).astype(float).to_numpy()
        return np.sort(sums)

    actual_sums = get_sorted_numeric_sums(actual)
    expected_sums = get_sorted_numeric_sums(expected)

    if len(actual_sums) != len(expected_sums):
        return False, {
            "reason": "Different number of numeric columns after exclusions",
            "actual_sums": actual_sums.tolist(),
            "expected_sums": expected_sums.tolist(),
        }

    match = np.allclose(actual_sums, expected_sums, atol=atol, rtol=rtol)

    return match, {
        "actual_sums": actual_sums.tolist(),
        "expected_sums": expected_sums.tolist(),
        "atol": atol,
        "rtol": rtol,
    }

#### Now, loop through all the cases, where we have generated some results. Compare with expected results.  

In [17]:
error_logs = []
test_case_errors = []
for test_case_result in test_case_results:
    test_case_error = {}
    test_case_error["test_results"] = test_case_result.copy()
    response = test_case_result["response"]
    error_dict = {"id": test_case_result["id"], "question": test_case_result["question"]}
    
    # Collecting error information
    # First we run the golden standard SQL:
    golden_sql = test_case_result.get("gold_sql", None)
    if golden_sql is not None:
        try:
            golden_result = conn.execute(golden_sql).fetchdf()
        except Exception as e:
            print(f"Error executing golden SQL: {e}")
            golden_result = None
    test_case_error["golden_result"] = golden_result

    error_dict["predicted category"] = test_case_result["category"]
    error_dict["actual chart type"] = test_case_result.get("chart_type", None)

    if response["success"]:
        error_dict["actual category"] = "answerable"

        result_df = response.get("dataframe", None)
        #error_dict["result correct"] = result_df.sort_index(axis=1).equals(golden_result.sort_index(axis=1))
        match, df_compare_dict = compare_by_numeric_sums_ignore_names(result_df, golden_result)
        if match: 
            error_dict["result correct"] = True
            #correct_result += 1
        else:
            error_dict["result correct"] = False
            #wrong_result += 1
        
        chart_response_dict = response.get("chart_response_dict", None)
        if chart_response_dict is not None: 
            error_dict["predicted chart type"] = chart_response_dict.get("chart_type", None)
            #error_dict["predicted chart x"] = chart_response_dict.get("x", None)
            #error_dict["predicted chart y"] = chart_response_dict.get("y", None)

    else:
        error_dict["actual category"] = response["error type"].lower()

    if error_dict.get("predicted chart type", None) == error_dict.get("actual chart type", None):
        error_dict["chart type correct"] = True
    else: 
        error_dict["chart type correct"] = False

    if error_dict["predicted category"] == error_dict["actual category"]:
        error_dict["category correct"] = True
    else:
        error_dict["category correct"] = False
    
    test_case_errors.append(test_case_error)
    error_logs.append(error_dict)


In [ ]:
error_logs_df = pd.DataFrame(error_logs)
#print(error_logs_df)
category_accuracy = error_logs_df["category correct"].mean()
result_accuracy = error_logs_df["result correct"].mean()
chart_type_accuracy = error_logs_df["chart type correct"].mean()
print(f"Category accuracy: {category_accuracy:.2%}, ({error_logs_df["category correct"].sum()} correct, {(1-error_logs_df["category correct"]).sum()} wrong)")
print(f"Result accuracy: {result_accuracy:.2%}, ({error_logs_df["result correct"].sum()} correct, {(1-error_logs_df["result correct"]).sum()} wrong)")
print(f"Chart type accuracy: {chart_type_accuracy:.2%}, ({error_logs_df["chart type correct"].sum()} correct, {(1-error_logs_df["chart type correct"]).sum()} wrong)") 



Category accuracy: 90.00%, (9 correct, 1 wrong)
Result accuracy: 100.00%, (6 correct, 0 wrong)
Chart type accuracy: 80.00%, (8 correct, 2 wrong)


: 

In [9]:
# We can also inspect individual cases to understand the errors better
print(test_case_errors[4].keys())
print(test_case_errors[4]['golden_result'])
print(test_case_errors[4]["test_results"]['response']["dataframe"])

dict_keys(['test_results', 'golden_result'])
             product_name  revenue
0       Spicy Beef Jaffle   5984.0
1      BBQ Chicken Jaffle   5785.5
2  Mushroom & Brie Jaffle   5410.0
3           Veggie Jaffle   5184.0
4        Breakfast Jaffle   5130.0
             Product Name  Total Revenue
0       Spicy Beef Jaffle         5984.0
1      BBQ Chicken Jaffle         5785.5
2  Mushroom & Brie Jaffle         5410.0
3           Veggie Jaffle         5184.0
4        Breakfast Jaffle         5130.0
